# Gene Ontology analysis: retrieval-augmented generation

**Research Question** Can a small retrieval step over the Gene Ontology make a language model's answers checkable, and does it agree with a standard enrichment analysis on a human gene list?

| Context | NOTES |
|---|---|
| Question | Does retrieving the five closest Gene Ontology terms help a language model answer biology questions with verifiable identifiers, and how does the retrieval agree with an enrichment analysis? |
| Data | `go-basic.obo` (the ontology) and `HUMAN-uniprot.gaf.gz` (human gene annotations), both from the Gene Ontology website. The Gene Ontology site footer states the licence as Creative Commons Attribution 4.0. |
| Models | Embedding: `sentence-transformers/all-MiniLM-L6-v2`. Language model, one swappable function: Claude (application programming interface) or a small open model run locally with PyTorch (`Qwen/Qwen2.5-0.5B-Instruct`). |
| Metrics | Retrieval: how often the known correct term is among the five results. Answers: citation check (are all cited identifiers among the retrieved ones, do they exist). Gene list: overlap with the enrichment answer key, mean and standard deviation over seeds 0, 1 and 42. |
| Time budget | About one day. Everything is cached, so **Run All** is fast after the first run. |

**What is retrieval and what is the language model?** Retrieval (Steps 4 to 7 and 9.1) uses no language model: it only embeds text and compares numbers. The language model (Steps 8 and 9.2) only writes the answer from the terms it is given. Every result below is labelled with the part it belongs to.

**Facts checked against sources while writing this notebook**
- The human annotation file moved in the Gene Ontology release of 2026-06-19: the new file is `HUMAN-uniprot.gaf.gz` in the `annotations/gaf/` folder of `current.geneontology.org`; the older name `goa_human.gaf.gz` stops working after September 2026 (announcement on the Alliance of Genome Resources community forum, confirmed by the Gene Ontology downloads table).
- The annotation file is in Gene Annotation Format 2.2: 17 tab-separated columns, header lines start with `!`, column 3 is the gene symbol, column 4 the relation (may start with `NOT`), column 5 the term identifier (Gene Ontology guide for the format).
- `go-basic.obo` is the version filtered so that the ontology has no cycles, so annotations can be propagated up the graph (Gene Ontology download page).
- Claude model identifiers `claude-sonnet-5-5` and `claude-haiku-5-5` are the ones listed on the current model overview page of the Claude documentation. Older cached copies of that documentation list other names, so if a call fails with "model not found", check that page.
- The embedding size, the text length limit and the number of parameters of the local model are **printed by the packages** in the steps below, not written from memory.
- The correct term for every test question is looked up by its exact **name** in the file you downloaded, and the notebook prints the identifier it found, so you can check each one.

---

## Step 1. Install, imports, versions and settings

In [1]:
# installing everything within the requirements notepad document
%pip install -r 00_requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing packages
import collections
import ipykernel
import getpass
import gzip
import importlib.metadata
import json
import math
import os
import platform
import random
import re
import sys
import urllib.request

import anthropic
import numpy
import pandas
import scipy.stats
import torch
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

C:\Users\evapa\Documents\RAG_using_GeneOntology\RAG_using_GeneOntology\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# What: prints the versions and writes them back to the requirements file, 
# and prints everything back here for documentation purposes
PACKAGE_NAMES = ["torch", "numpy", "pandas", "scipy", "sentence-transformers",
                 "transformers", "huggingface_hub", "anthropic"]

print("Python:", sys.version.split()[0])
print("Operating system:", platform.platform())
requirement_lines = []
for package_name in PACKAGE_NAMES:
    package_version = importlib.metadata.version(package_name).split("+")[0]
    print(package_name + ":", package_version)
    requirement_lines.append(package_name + "==" + package_version)

with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write("\n".join(requirement_lines) + "\n")
print("requirements.txt now holds the exact versions listed above.")

Python: 3.12.0
Operating system: Windows-11-10.0.26300-SP0
torch: 2.15.0
numpy: 2.5.5
pandas: 3.1.1
scipy: 1.18.1
sentence-transformers: 6.3.0
transformers: 5.23.0
huggingface_hub: 2.4.0
anthropic: 2.10.0
requirements.txt now holds the exact versions listed above.


In [4]:
# Parameter Setup
DATA_FOLDER = "data"
RESULTS_FOLDER = "results"
os.makedirs(DATA_FOLDER, exist_ok=True)
os.makedirs(RESULTS_FOLDER, exist_ok=True)

ONTOLOGY_PATH = os.path.join(DATA_FOLDER, "go-basic.obo")
ANNOTATION_PATH = os.path.join(DATA_FOLDER, "HUMAN-uniprot.gaf.gz")
ONTOLOGY_ADDRESSES = [
    "https://purl.obolibrary.org/obo/go/go-basic.obo",
    "https://current.geneontology.org/ontology/go-basic.obo",
]
ANNOTATION_ADDRESSES = [
    "https://current.geneontology.org/annotations/gaf/HUMAN-uniprot.gaf.gz",
]
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LOCAL_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
CLAUDE_MODEL_NAME = "claude-sonnet-5-5"          # a cheaper choice is "claude-haiku-5-5"

BACKENDS_TO_USE = ["claude", "local"]            # remove one word to use only one backend
HOW_MANY_TERMS = 5                               # terms retrieved for every question
MAXIMUM_NEW_TOKENS_LOCAL = 300
TERM_LIMIT = None                                # for a quick trial use 3000 (searches only those terms)
RANDOM_SEEDS = [0, 1, 42]
ANSWER_CACHE_PATH = os.path.join(DATA_FOLDER, "answer_cache.json")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Computing on:", device)
print("Folders:", DATA_FOLDER, "and", RESULTS_FOLDER)

summary_lines = []                               # every key result is also saved to results/summary.txt


# HELPER FUNCTION
def report(text):
    # Prints a result line and keeps it for the summary file.
    print(text)
    summary_lines.append(text)


# HELPER FUNCTION
def set_seed(seed_value):
    # Fixes all random number generators so that a run can be repeated.
    random.seed(seed_value)
    numpy.random.seed(seed_value)
    torch.manual_seed(seed_value)

Computing on: cpu
Folders: data and results


## Step 2. Download the data and prove access

This step downloads the two files (skipped if they are already in `data/`), then checks that the two model sites can be reached and prints what the files contain. The Gene Ontology site asks users to record the **ontology version** and the **date in the annotation file header**, so both are printed here.

**If a line says `FAILED`:** download the file in your browser from https://geneontology.org/docs/download-ontology/ or https://geneontology.org/docs/download-go-annotations/downloads/ (human, UniProt-centric file) and drop it into the `data` folder in the Explorer panel.

In [5]:
# Let's download the two files if they are missing and checks the two model sites.
download_opener = urllib.request.build_opener()
download_opener.addheaders = [("User-Agent", "Mozilla/5.0")]
urllib.request.install_opener(download_opener)

# Expected output: four result lines. Nothing should say FAILED.
# HELPER FUNCTION
def download_if_missing(path, addresses):
    # Returns (status, detail). Tries each address in order.
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return "already on disk", str(round(os.path.getsize(path) / 1e6, 1)) + " megabytes"
    errors = []
    for address in addresses:
        try:
            urllib.request.urlretrieve(address, path)
            return "downloaded", address + " | " + str(round(os.path.getsize(path) / 1e6, 1)) + " megabytes"
        except Exception as error:
            errors.append(address + " -> " + type(error).__name__ + ": " + str(error)[:120])
            if os.path.exists(path):
                os.remove(path)
    return "FAILED", " || ".join(errors)


for item, path, addresses in [("go-basic.obo", ONTOLOGY_PATH, ONTOLOGY_ADDRESSES),
                              ("HUMAN-uniprot.gaf.gz", ANNOTATION_PATH, ANNOTATION_ADDRESSES)]:
    status, detail = download_if_missing(path, addresses)
    report(item + ": " + status + " | " + detail)

for model_name in [EMBEDDING_MODEL_NAME, LOCAL_MODEL_NAME]:
    try:
        hf_hub_download(repo_id=model_name, filename="config.json")
        report(model_name + ": reachable")
    except Exception as error:
        report(model_name + ": FAILED | " + type(error).__name__ + ": " + str(error)[:150])

for path in [ONTOLOGY_PATH, ANNOTATION_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError("Missing file: " + path + ". Download it by hand (see the text above) into the data folder.")

go-basic.obo: already on disk | 32.2 megabytes
HUMAN-uniprot.gaf.gz: already on disk | 12.1 megabytes


sentence-transformers/all-MiniLM-L6-v2: reachable


Qwen/Qwen2.5-0.5B-Instruct: reachable


In [6]:
# a look inside both files (version, date and format).
print("go-basic.obo, first header lines:")
with open(ONTOLOGY_PATH, encoding="utf-8") as file:
    for line_number, line in enumerate(file):
        if line.startswith("format-version") or line.startswith("data-version") or line.startswith("ontology"):
            report("  " + line.rstrip())
        if line_number > 12:
            break

print("HUMAN-uniprot.gaf.gz, header lines and first data lines:")
shown_data_lines = 0
with gzip.open(ANNOTATION_PATH, "rt", encoding="utf-8") as file:
    for line in file:
        if line.startswith("!gaf-version") or line.startswith("!date-generated") or line.startswith("!generated-by") or line.startswith("!go-version"):
            report("  " + line.rstrip())
        elif not line.startswith("!"):
            columns = line.rstrip("\n").split("\t")
            print("  columns in this line:", len(columns), "| gene symbol:", columns[2], "| relation:", columns[3],
                  "| term identifier:", columns[4], "| evidence:", columns[6])
            shown_data_lines += 1
            if shown_data_lines >= 2:
                break

go-basic.obo, first header lines:
  format-version: 1.2
  data-version: releases/2026-07-26
HUMAN-uniprot.gaf.gz, header lines and first data lines:
  !gaf-version: 2.2
  !generated-by: UniProt
  !date-generated: 2026-07-28 19:42
  !go-version: http://purl.obolibrary.org/obo/go/releases/2026-07-26/extensions/go-plus.ofn
  columns in this line: 17 | gene symbol: NUDT4B | relation: enables | term identifier: GO:0005515 | evidence: IPI
  columns in this line: 17 | gene symbol: NUDT4B | relation: enables | term identifier: GO:0008486 | evidence: IEA


## Step 3. Basic data checks

This step reads the ontology into a table and the annotations into a gene-to-term dictionary, and records what is found:
1. Obsolete terms are removed (the ontology marks them `is_obsolete: true`).
2. Every term keeps its identifier, name, branch (**namespace**), definition and **parent terms** (`is_a` and `relationship: part_of` lines). The parents are needed for the enrichment analysis in Step 10.
3. In the annotation file, lines whose relation contains `NOT` are skipped (they say a gene does **not** have the function), and lines whose term is not in the ontology table (obsolete or alternative identifiers) are skipped and counted.
4. Each annotation is **propagated upward**: a gene linked to a specific term also counts for every ancestor term. `go-basic.obo` is built so that this is safe.

In [7]:
def read_ontology_terms(path):
    terms = []
    current = None
    with open(path, encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line == "[Term]":
                if current is not None:
                    terms.append(current)
                current = {"identifier": "", "name": "", "namespace": "", "definition": "",
                           "obsolete": False, "parent_identifiers": []}
            elif line.startswith("[") and line.endswith("]"):
                # another kind of block (for example [Typedef]); finish the current term
                if current is not None:
                    terms.append(current)
                current = None
            elif current is not None:
                if line.startswith("id: "):
                    current["identifier"] = line[4:]
                elif line.startswith("name: "):
                    current["name"] = line[6:]
                elif line.startswith("namespace: "):
                    current["namespace"] = line[11:]
                elif line.startswith("def: "):
                    # format:  def: "the definition text." [reference list]
                    first_quote = line.find('"')
                    last_quote = line.rfind('"')
                    current["definition"] = line[first_quote + 1:last_quote]
                elif line.startswith("is_obsolete: true"):
                    current["obsolete"] = True
                elif line.startswith("is_a: "):
                    current["parent_identifiers"].append(line.split()[1])
                elif line.startswith("relationship: part_of "):
                    current["parent_identifiers"].append(line.split()[2])
    if current is not None:
        terms.append(current)
    return pandas.DataFrame(terms)


all_terms = read_ontology_terms(ONTOLOGY_PATH)
terms = all_terms[~all_terms["obsolete"]].reset_index(drop=True)
terms["text"] = terms["name"] + ". " + terms["definition"]

report("Terms read: " + str(len(all_terms)) + " | obsolete terms removed: " + str(int(all_terms["obsolete"].sum())) + " | terms kept: " + str(len(terms)))
for namespace_name, count in terms["namespace"].value_counts().items():
    report("  terms in " + namespace_name + ": " + str(count))
report("Terms without a definition: " + str(int((terms["definition"] == "").sum())))
report("Terms with no parent term: " + str(int(terms["parent_identifiers"].apply(len).eq(0).sum())))
report("Terms whose name appears more than once: " + str(int(terms["name"].duplicated(keep=False).sum())))
report("Terms whose definition is identical to another term's definition: " + str(int((terms["definition"].duplicated(keep=False) & (terms["definition"] != "")).sum())))

Terms read: 48340 | obsolete terms removed: 10248 | terms kept: 38092
  terms in biological_process: 23974
  terms in molecular_function: 10041
  terms in cellular_component: 4077
Terms without a definition: 0
Terms with no parent term: 3
Terms whose name appears more than once: 0
Terms whose definition is identical to another term's definition: 0


In [8]:
# This step calculates the number of ancestors of one example term
# through lookup tables and a function that lists every ancestor of each term.
#
term_by_identifier = {row["identifier"]: row for row in terms.to_dict("records")}
parent_identifiers = {identifier: row["parent_identifiers"] for identifier, row in term_by_identifier.items()}
ancestor_cache = {}


# HELPER FUNCTION
def ancestors_of(identifier):
    # All terms above this term, following the parent links upward (the term itself is not included).
    if identifier in ancestor_cache:
        return ancestor_cache[identifier]
    found = set()
    to_visit = list(parent_identifiers.get(identifier, []))
    while to_visit:
        current = to_visit.pop()
        if current in found:
            continue
        found.add(current)
        to_visit.extend(parent_identifiers.get(current, []))
    ancestor_cache[identifier] = found
    return found


# HELPER FUNCTION
def find_identifier_by_name(exact_name):
    # Looks up a term by its exact name; returns None if there is no such term.
    matches = terms[terms["name"] == exact_name]
    if len(matches) == 0:
        return None
    biological_matches = matches[matches["namespace"] == "biological_process"]
    return (biological_matches if len(biological_matches) > 0 else matches).iloc[0]["identifier"]


example_identifier = find_identifier_by_name("autophagy")
print("Example term:", example_identifier, "|", term_by_identifier[example_identifier]["name"] if example_identifier else "not found")
if example_identifier:
    print("Number of ancestors of the example term:", len(ancestors_of(example_identifier)))

Example term: GO:0006914 | autophagy
Number of ancestors of the example term: 5


In [9]:
# Here we read the human annotation file and propagate every annotation to the ancestor terms.
gene_to_terms = collections.defaultdict(set)
lines_read = 0
skipped_negative = 0
skipped_unknown_term = 0
taxon_counts = collections.Counter()

with gzip.open(ANNOTATION_PATH, "rt", encoding="utf-8") as file:
    for line in file:
        if line.startswith("!"):
            continue
        columns = line.rstrip("\n").split("\t")
        lines_read += 1
        taxon_counts[columns[12]] += 1
        if "NOT" in columns[3]:
            skipped_negative += 1
            continue
        term_identifier = columns[4]
        if term_identifier not in term_by_identifier:
            skipped_unknown_term += 1
            continue
        gene_symbol = columns[2]
        gene_to_terms[gene_symbol].add(term_identifier)
        gene_to_terms[gene_symbol].update(ancestors_of(term_identifier))

term_to_genes = collections.defaultdict(set)
for gene_symbol, term_set in gene_to_terms.items():
    for term_identifier in term_set:
        term_to_genes[term_identifier].add(gene_symbol)

report("Annotation lines read: " + str(lines_read))
report("Skipped, negative relation (NOT): " + str(skipped_negative) + " | skipped, term not in the ontology table: " + str(skipped_unknown_term))
report("Genes with at least one annotation (used as the background): " + str(len(gene_to_terms)))
report("Terms with at least one gene after propagation: " + str(len(term_to_genes)))
report("Most common values in the taxon column: " + str(taxon_counts.most_common(3)))

Annotation lines read: 910476
Skipped, negative relation (NOT): 1497 | skipped, term not in the ontology table: 0
Genes with at least one annotation (used as the background): 38810
Terms with at least one gene after propagation: 21102
Most common values in the taxon column: [('taxon:9606', 909626), ('taxon:9606|taxon:562', 92), ('taxon:9606|taxon:1280', 83)]


## Step 4. Data into PyTorch: one embedding per term (cached)

In this step, the embedding model turns each term's text (name and definition) into a list of numbers, an **embedding**. Texts with similar meaning get similar numbers. The result is one PyTorch table with one row per term. The first run downloads the model and takes roughly 5 to 10 minutes on an ordinary processor; later runs load the cache. The cache also stores the identifiers, so a changed ontology file can never be paired with an old cache.

In [10]:
if TERM_LIMIT is None:
    searchable_terms = terms
    EMBEDDING_PATH = os.path.join(DATA_FOLDER, "go_term_embeddings.npz")
else:
    searchable_terms = terms.iloc[:TERM_LIMIT].reset_index(drop=True)
    EMBEDDING_PATH = os.path.join(DATA_FOLDER, "go_term_embeddings_trial.npz")

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
report("Embedding size (printed by the package): " + str(embedding_model.get_sentence_embedding_dimension()))
report("Text length limit in word pieces (printed by the package): " + str(embedding_model.max_seq_length))

token_counts = [len(ids) for ids in embedding_model.tokenizer(searchable_terms["text"].tolist())["input_ids"]]
report("Terms with a text longer than the limit (the end is cut off): " + str(sum(count > embedding_model.max_seq_length for count in token_counts)))

searchable_identifiers = searchable_terms["identifier"].tolist()
cache_is_valid = False
if os.path.exists(EMBEDDING_PATH):
    saved = numpy.load(EMBEDDING_PATH)
    if saved["identifiers"].tolist() == searchable_identifiers:
        term_embeddings_array = saved["embeddings"]
        cache_is_valid = True
        print("Loaded the cached embeddings.")
    else:
        print("The cache belongs to a different ontology file; computing again.")

if not cache_is_valid:
    term_embeddings_array = embedding_model.encode(
        searchable_terms["text"].tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True
    )
    numpy.savez(EMBEDDING_PATH, embeddings=term_embeddings_array, identifiers=numpy.array(searchable_identifiers))

term_embeddings = torch.from_numpy(term_embeddings_array).float()      # one row per term
print("Shape of one item:", tuple(term_embeddings[0].shape))
print("Shape of the whole table (terms, numbers per term):", tuple(term_embeddings.shape))
print("Length of the first row (should be 1.0):", round(float(numpy.linalg.norm(term_embeddings_array[0])), 4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1974.98it/s]

C:\Users\evapa\AppData\Local\Temp\ipykernel_12016\1680821422.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  report("Embedding size (printed by the package): " + str(embedding_model.get_sentence_embedding_dimension()))


Embedding size (printed by the package): 384
Text length limit in word pieces (printed by the package): 256


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (327 > 256). Running this sequence through the model will result in indexing errors


Terms with a text longer than the limit (the end is cut off): 30
Loaded the cached embeddings.
Shape of one item: (384,)
Shape of the whole table (terms, numbers per term): (38092, 384)
Length of the first row (should be 1.0): 1.0


## Step 5. Retrieval: the five closest terms

In the retrieval stage, the question is embedded the same way, and one matrix multiplication gives its closeness (**cosine similarity**) to every term. `torch.topk` returns the five largest. The optional `namespace` setting limits the search to one branch (`biological_process`, `molecular_function` or `cellular_component`).

In [11]:
# Below the search function is defined as a helper. PyTorch does the similarity calculation.
namespace_of_each_term = searchable_terms["namespace"].to_numpy()


# HELPER FUNCTION
def search_terms(question, how_many=HOW_MANY_TERMS, namespace=None):
    question_embedding = torch.from_numpy(
        embedding_model.encode([question], normalize_embeddings=True)[0]
    ).float()
    similarities = term_embeddings @ question_embedding                    # one number per term
    if namespace is not None:
        outside_namespace = torch.from_numpy(namespace_of_each_term != namespace)
        similarities = similarities.masked_fill(outside_namespace, -1.0)   # -1 is the lowest possible score
    best_values, best_positions = torch.topk(similarities, k=how_many)
    found = searchable_terms.iloc[best_positions.tolist()][["identifier", "name", "definition"]].copy()
    found["similarity"] = best_values.tolist()
    return found

In [12]:
# This cell checks the search function and takes a first look at three questions, including finding the shape.
frozen_numbers = sum(parameter.numel() for parameter in embedding_model.parameters())
print("Learnable numbers in the retrieval step: 0 | frozen numbers in the embedding model:", frozen_numbers)
check_result = search_terms("how cells recycle their own components")
print("Shape of one search result (rows, columns):", check_result.shape)

for question in ["ageing of cells", "immune response against bacteria", "chromatin remodelling"]:
    print("QUESTION:", question)
    found = search_terms(question, how_many=3)
    for _, row in found.iterrows():
        print("  " + row["identifier"] + " | " + row["name"] + " | similarity " + str(round(row["similarity"], 3)))

Learnable numbers in the retrieval step: 0 | frozen numbers in the embedding model: 22713216


Shape of one search result (rows, columns):

 (5, 4)
QUESTION: ageing of cells
  GO:0090398 | cellular senescence | similarity 0.706
  GO:0044838 | cell quiescence | similarity 0.626
  GO:0090400 | stress-induced premature senescence | similarity 0.597
QUESTION: immune response against bacteria
  GO:0140367 | antibacterial innate immune response | similarity 0.736
  GO:0042742 | defense response to bacterium | similarity 0.675
  GO:0019731 | antibacterial humoral response | similarity 0.664
QUESTION: chromatin remodelling
  GO:0006338 | chromatin remodeling | similarity 0.757
  GO:0140861 | DNA repair-dependent chromatin remodeling | similarity 0.723
  GO:0006325 | chromatin organization | similarity 0.718


## Step 6. Baselines for retrieval
NOTE: a score means little without something to compare it with. Two do-nothing baselines:
1. **Chance:** a random choice of five terms. The chance that one named term is among them is five divided by the number of terms.
2. **Naive word search:** counts how many words of the question appear in a term's text. It knows nothing about meaning and cannot match different spellings.

In [13]:
# HELPER FUNCTION; that defines the naive word search and prints the chance level.

def naive_word_search(question, how_many=HOW_MANY_TERMS):
    # Counts how many question words appear in the term text. No meaning, only exact words.
    question_words = set(question.lower().replace("?", "").split())
    counts = searchable_terms["text"].str.lower().apply(lambda text: sum(word in text for word in question_words))
    best_positions = numpy.argsort(-counts.to_numpy(), kind="stable")[:how_many]
    return searchable_terms.iloc[best_positions]


report("Chance that one named term is among five random terms: " + str(round(HOW_MANY_TERMS / len(searchable_terms), 5)))
print("Naive word search returns", len(naive_word_search("how cells recycle their own components")), "terms.")

Chance that one named term is among five random terms: 0.00013


Naive word search returns 5 terms.


## Step 7. The language model part: one swappable function and a citation check

In the code, the command `generate_answer(prompt, backend)` is the only place that knows which language model is used. Everything else calls this function, so a third backend can be added later without changing any other cell.

| Backend | Quality | Cost | Speed |
|---|---|---|---|
| `"claude"` | Best | A few cents for the whole notebook | Fast |
| `"local"` | Weaker, often ignores the rules | Free | Slow on a processor; the first call downloads the model |

Answers are saved in `data/answer_cache.json`, so a re-run costs nothing and returns the same text. (Without the cache a language model answers a little differently every time.)

Rule: every identifier in the answer must be one of the five retrieved identifiers. The check also records whether each cited identifier exists in the ontology at all. An identifier that does not exist is invented.

**The Claude key** (if used) should be typed into a hidden box, kept only in the memory of the notebook, and never written to a file. If the environment variable `ANTHROPIC_API_KEY` is set it is used without asking. Press Enter on an empty box to use only the local backend.

In [14]:
# This cell loads the answer cache and asks for the Claude key.

if os.path.exists(ANSWER_CACHE_PATH):
    with open(ANSWER_CACHE_PATH, encoding="utf-8") as file:
        answer_cache = json.load(file)
else:
    answer_cache = {}
print("Answers already cached:", len(answer_cache))

claude_key = os.environ.get("ANTHROPIC_API_KEY", "")
if claude_key == "" and "claude" in BACKENDS_TO_USE:
    try:
        claude_key = getpass.getpass("Claude key (press Enter to skip the Claude backend): ").strip()
    except Exception:
        # A run from the terminal has no box to type into: the Claude backend is skipped.
        claude_key = ""
        print("No key box is available in this run and ANTHROPIC_API_KEY is not set. The Claude backend is skipped.")
if claude_key == "":
    BACKENDS_TO_USE = [backend for backend in BACKENDS_TO_USE if backend != "claude"]
print("Backends that will be used:", BACKENDS_TO_USE)

Answers already cached: 25
No key box is available in this run and ANTHROPIC_API_KEY is not set. The Claude backend is skipped.
Backends that will be used: ['local']


In [15]:
# Defining the the two backend functions and the one swappable function.

local_tokenizer = None
local_model = None

# HELPER FUNCTION
def generate_with_claude(prompt):
    client = anthropic.Anthropic(api_key=claude_key)
    response = client.messages.create(
        model=CLAUDE_MODEL_NAME,
        max_tokens=600,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text


# HELPER FUNCTION
def generate_with_local_model(prompt):
    global local_tokenizer, local_model
    if local_model is None:                                  # load only once
        local_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME)
        local_model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL_NAME).to(device)
        local_model.eval()
        print("Numbers in the local model (printed by the package):", sum(parameter.numel() for parameter in local_model.parameters()))
    chat_text = local_tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )
    inputs = local_tokenizer(chat_text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = local_model.generate(**inputs, max_new_tokens=MAXIMUM_NEW_TOKENS_LOCAL, do_sample=False)
    new_pieces = output[0][inputs["input_ids"].shape[1]:]
    return local_tokenizer.decode(new_pieces, skip_special_tokens=True).strip()


# HELPER FUNCTION
def generate_answer(prompt, backend):
    # THE swappable function: the same call for every backend, with a cache on disk.
    model_name = CLAUDE_MODEL_NAME if backend == "claude" else LOCAL_MODEL_NAME
    cache_key = backend + "||" + model_name + "||" + prompt
    if cache_key in answer_cache:
        return answer_cache[cache_key]
    if backend == "claude":
        answer = generate_with_claude(prompt)
    elif backend == "local":
        answer = generate_with_local_model(prompt)
    else:
        raise ValueError("Unknown backend: " + backend)
    answer_cache[cache_key] = answer
    try:
        with open(ANSWER_CACHE_PATH, "w", encoding="utf-8") as file:
            json.dump(answer_cache, file)
    except OSError as error:
        report("Cache file could not be written (" + str(error)[:100] + "). Continuing without saving it.")
    return answer

In [16]:
# one tiny test call for each backend, to prove that it works.
for backend in list(BACKENDS_TO_USE):
    try:
        print(backend, "->", generate_answer("Reply with the single word: ready", backend))
    except Exception as error:
        print(backend, "FAILED:", type(error).__name__, str(error)[:300])
        BACKENDS_TO_USE = [name for name in BACKENDS_TO_USE if name != backend]
report("Backends that work: " + str(BACKENDS_TO_USE))

local -> waiting
Backends that work: ['local']


In [17]:
# Defining the two prompt builders and the citation checker.
# HELPER FUNCTION
def build_prompt_with_retrieval(question, found):
    lines = []
    for _, row in found.iterrows():
        lines.append("- " + row["identifier"] + " | " + row["name"] + ": " + row["definition"])
    return (
        "You are helping a biology student.\n"
        "Answer the question below using ONLY the Gene Ontology terms listed. "
        "After every statement, cite the identifier of the term it comes from, written exactly like GO:0000000. "
        "Do not mention any identifier that is not in the list. "
        "If the listed terms are not enough to answer, say so clearly. "
        "Keep the answer under 120 words.\n\n"
        "Question: " + question + "\n\n"
        "Gene Ontology terms:\n" + "\n".join(lines) + "\n"
    )


# HELPER FUNCTION
def build_prompt_without_retrieval(question):
    return (
        "You are helping a biology student.\n"
        "Answer the question below from your own knowledge. "
        "After every statement, cite the identifier of the Gene Ontology term it relates to, "
        "written exactly like GO:0000000. "
        "If you are not sure of an identifier, say so instead of guessing. "
        "Keep the answer under 120 words.\n\n"
        "Question: " + question + "\n"
    )


IDENTIFIER_PATTERN = re.compile(r"GO:\d{7}")        # how every identifier is written


# HELPER FUNCTION
def check_citations(answer, retrieved_identifiers):
    cited = sorted(set(IDENTIFIER_PATTERN.findall(answer)))
    inside_retrieved = [identifier for identifier in cited if identifier in retrieved_identifiers]
    outside_retrieved = [identifier for identifier in cited if identifier not in retrieved_identifiers]
    existing = [identifier for identifier in cited if identifier in term_by_identifier]
    not_existing = [identifier for identifier in cited if identifier not in term_by_identifier]
    return {
        "cited": cited,
        "inside_retrieved": inside_retrieved,
        "outside_retrieved": outside_retrieved,
        "existing_in_ontology": existing,
        "not_in_ontology": not_existing,
        "passes": len(cited) > 0 and len(outside_retrieved) == 0,
    }


demonstration_found = search_terms("How do cells recycle their own components?")
demonstration = check_citations(
    "One valid [" + demonstration_found.iloc[0]["identifier"] + "], one outside the list GO:0000001, one that does not exist GO:9999999.",
    demonstration_found["identifier"].tolist(),
)
print("Demonstration of the check on a made-up answer:", demonstration)

Demonstration of the check on a made-up answer: {'cited': ['GO:0000001', 'GO:0043954', 'GO:9999999'], 'inside_retrieved': ['GO:0043954'], 'outside_retrieved': ['GO:0000001', 'GO:9999999'], 'existing_in_ontology': ['GO:0000001', 'GO:0043954'], 'not_in_ontology': ['GO:9999999'], 'passes': False}


In [18]:
# a first run on one example question, for every working backend, with and without retrieval.
example_question = "Which biological processes link chromatin structure to gene activity?"
example_found = search_terms(example_question)
print("QUESTION:", example_question)
for _, row in example_found.iterrows():
    print("  retrieved: " + row["identifier"] + " | " + row["name"])
for backend in BACKENDS_TO_USE:
    for condition in ["with retrieval", "without retrieval"]:
        prompt = build_prompt_with_retrieval(example_question, example_found) if condition == "with retrieval" else build_prompt_without_retrieval(example_question)
        answer = generate_answer(prompt, backend)
        checked = check_citations(answer, example_found["identifier"].tolist())
        print()
        print("[" + backend + ", " + condition + "] " + answer)
        print("  cited:", checked["cited"], "| not in the ontology:", checked["not_in_ontology"], "| outside the retrieved list:", checked["outside_retrieved"], "| passes:", checked["passes"])

QUESTION: Which biological processes link chromatin structure to gene activity?
  retrieved: GO:0030527 | structural constituent of chromatin
  retrieved: GO:0006325 | chromatin organization
  retrieved: GO:1902275 | regulation of chromatin organization
  retrieved: GO:0031490 | chromatin DNA binding
  retrieved: GO:0000785 | chromatin

[local, with retrieval] The biological processes linking chromatin structure to gene activity include:

- **GO:0030527**: Structural Constituent of Chromatin - This term indicates the action of a molecule contributing to the structural integrity of chromatin.
  
- **GO:0006325**: Chromatin Organization - This term describes the assembly or remodeling of chromatin composed of DNA complexed with histones, other associated proteins, and sometimes RNA.
  
- **GO:1902275**: Regulation of Chromatin Organization - This term refers to any process that modulates the frequency, rate, or extent of chromatin organization.
  
- **GO:0031490**: Chromatin DNA Binding 

## Step 8. Test once: retrieval quality and the effect of retrieval on answers

The question set below was written **before** looking at any result and is not changed afterwards. Each question has one known correct term, given by its exact name; the identifier is looked up in your file and printed. Some questions avoid the words of the term name, and one uses British spelling on purpose.

### 8.1 Retrieval only (no language model)
A question is a **hit** if the correct term is among the five results. A **related hit** also accepts a result that is an ancestor or a descendant of the correct term (a looser measure). The embedding search is compared with the two baselines from Step 6.

In [19]:
# Defining the fixed question set (question, exact name of the correct term) and the identifier lookup.
test_questions = [
    ("How do cells digest and recycle their own components?", "autophagy"),
    ("How do cells repair damaged genetic material?", "DNA repair"),
    ("What is the process of programmed cell death?", "apoptotic process"),
    ("How do cells get older?", "aging"),
    ("How do cells respond to oxidative stress?", "response to oxidative stress"),
    ("How is insulin released from pancreatic cells?", "insulin secretion"),
    ("How do neurons pass signals across a synapse?", "chemical synaptic transmission"),
    ("chromatin remodelling", "chromatin remodeling"),                  # British spelling in the question, American in the ontology
    ("How does the body defend itself against bacteria?", "defense response to bacterium"),
    ("How is a methyl group added to genetic material?", "DNA methylation"),
]

known_questions = []
for question, correct_name in test_questions:
    correct_identifier = find_identifier_by_name(correct_name)
    if correct_identifier is None:
        print("SKIPPED (name not found in the ontology file):", correct_name)
        continue
    known_questions.append((question, correct_name, correct_identifier))
    print(correct_identifier + " | " + correct_name + "  <-  " + question)
report("Questions with a known correct term: " + str(len(known_questions)) + " of " + str(len(test_questions)))


# HELPER FUNCTION
def terms_are_related(first_identifier, second_identifier):
    # True if the two terms are the same, or one is an ancestor of the other.
    return (first_identifier == second_identifier
            or second_identifier in ancestors_of(first_identifier)
            or first_identifier in ancestors_of(second_identifier))

GO:0006914 | autophagy  <-  How do cells digest and recycle their own components?
GO:0006281 | DNA repair  <-  How do cells repair damaged genetic material?
GO:0006915 | apoptotic process  <-  What is the process of programmed cell death?
SKIPPED (name not found in the ontology file): aging
GO:0006979 | response to oxidative stress  <-  How do cells respond to oxidative stress?
GO:0030073 | insulin secretion  <-  How is insulin released from pancreatic cells?
GO:0007268 | chemical synaptic transmission  <-  How do neurons pass signals across a synapse?
GO:0006338 | chromatin remodeling  <-  chromatin remodelling
GO:0042742 | defense response to bacterium  <-  How does the body defend itself against bacteria?
SKIPPED (name not found in the ontology file): DNA methylation
Questions with a known correct term: 8 of 10


In [20]:
# Measuring how often the correct term is among the five results, for the embedding search and the naive search.
retrieval_rows = []
for question, correct_name, correct_identifier in known_questions:
    embedding_found = search_terms(question)
    naive_found = naive_word_search(question)
    embedding_identifiers = embedding_found["identifier"].tolist()
    naive_identifiers = naive_found["identifier"].tolist()
    row = {
        "question": question, "correct_term": correct_name,
        "embedding_top_one": embedding_identifiers[0] == correct_identifier,
        "embedding_hit": correct_identifier in embedding_identifiers,
        "embedding_related_hit": any(terms_are_related(identifier, correct_identifier) for identifier in embedding_identifiers),
        "naive_hit": correct_identifier in naive_identifiers,
        "naive_related_hit": any(terms_are_related(identifier, correct_identifier) for identifier in naive_identifiers),
    }
    retrieval_rows.append(row)
    print(("HIT   " if row["embedding_hit"] else "miss  ") + correct_name + " | top result: " + embedding_found.iloc[0]["name"])

retrieval_table = pandas.DataFrame(retrieval_rows)
retrieval_table.to_csv(os.path.join(RESULTS_FOLDER, "retrieval_per_question.csv"), index=False)
question_count = len(retrieval_table)
report("Embedding search: correct term first in " + str(int(retrieval_table["embedding_top_one"].sum())) + ", among the five in " + str(int(retrieval_table["embedding_hit"].sum())) + ", related among the five in " + str(int(retrieval_table["embedding_related_hit"].sum())) + " of " + str(question_count) + " questions")
report("Naive word search: correct term among the five in " + str(int(retrieval_table["naive_hit"].sum())) + ", related among the five in " + str(int(retrieval_table["naive_related_hit"].sum())) + " of " + str(question_count) + " questions")
report("Chance level for the correct term among five random terms, per question: " + str(round(HOW_MANY_TERMS / len(searchable_terms), 5)))

miss  autophagy | top result: digestion
HIT   DNA repair | top result: DNA repair


HIT   apoptotic process | top result: programmed cell death
HIT   response to oxidative stress | top result: cellular response to oxidative stress


miss  insulin secretion | top result: insulin catabolic process
miss  chemical synaptic transmission | top result: synaptic signaling


HIT   chromatin remodeling | top result: chromatin remodeling
HIT   defense response to bacterium | top result: defense response to bacterium


Embedding search: correct term first in 3, among the five in 5, related among the five in 7 of 8 questions
Naive word search: correct term among the five in 2, related among the five in 5 of 8 questions
Chance level for the correct term among five random terms, per question: 0.00013


### 8.2 Language model: answers with and without retrieval
Each question is answered by each working backend under two conditions. The retrieved set is computed for both conditions, but in the *without retrieval* condition the model never sees it; it only serves as a reference.

| Measure | Meaning |
|---|---|
| cited | how many different identifiers the answer cites |
| existing | how many of them exist in the ontology (the rest are invented) |
| inside retrieved | how many are among the five retrieved terms |
| passes | the strict check: at least one identifier, and every one is among the retrieved terms |
| relevant | at least one cited identifier is the correct term, or an ancestor or descendant of it (identifier looked up in the ontology, not taken from the answer text) |

In [21]:
# Running the comparison for every question, backend and condition.
comparison_rows = []
answer_records = []
for question, correct_name, correct_identifier in known_questions:
    found = search_terms(question)
    retrieved_identifiers = found["identifier"].tolist()
    for backend in BACKENDS_TO_USE:
        for condition in ["with retrieval", "without retrieval"]:
            prompt = build_prompt_with_retrieval(question, found) if condition == "with retrieval" else build_prompt_without_retrieval(question)
            answer = generate_answer(prompt, backend)
            checked = check_citations(answer, retrieved_identifiers)
            relevant = any(terms_are_related(identifier, correct_identifier) for identifier in checked["existing_in_ontology"])
            comparison_rows.append({
                "question": question, "backend": backend, "condition": condition,
                "cited": len(checked["cited"]), "existing": len(checked["existing_in_ontology"]),
                "inside_retrieved": len(checked["inside_retrieved"]),
                "passes": checked["passes"], "relevant": relevant,
            })
            answer_records.append({"question": question, "backend": backend, "condition": condition, "answer": answer, "checked": checked})

comparison_table = pandas.DataFrame(comparison_rows)
comparison_table.to_csv(os.path.join(RESULTS_FOLDER, "comparison_per_question.csv"), index=False)
with open(os.path.join(RESULTS_FOLDER, "answers_with_and_without_retrieval.txt"), "w", encoding="utf-8") as file:
    for record in answer_records:
        file.write("QUESTION: " + record["question"] + " | " + record["backend"] + " | " + record["condition"] + "\n")
        file.write(record["answer"] + "\n")
        file.write("CITATION CHECK: " + json.dumps(record["checked"]) + "\n\n")

for backend in BACKENDS_TO_USE:
    for condition in ["with retrieval", "without retrieval"]:
        part = comparison_table[(comparison_table["backend"] == backend) & (comparison_table["condition"] == condition)]
        report(backend + " | " + condition + " | answers: " + str(len(part))
               + " | identifiers cited: " + str(int(part["cited"].sum()))
               + " | existing: " + str(int(part["existing"].sum()))
               + " | inside retrieved: " + str(int(part["inside_retrieved"].sum()))
               + " | answers passing the strict check: " + str(int(part["passes"].sum()))
               + " | answers with a relevant term: " + str(int(part["relevant"].sum())))

local | with retrieval | answers: 8 | identifiers cited: 5 | existing: 5 | inside retrieved: 5 | answers passing the strict check: 1 | answers with a relevant term: 1


local | without retrieval | answers: 8 | identifiers cited: 0 | existing: 0 | inside retrieved: 0 | answers passing the strict check: 0 | answers with a relevant term: 0


## Step 9. Measuring near-duplicate terms

The ontology has many closely related terms, so the five retrieved terms may be almost the same idea. For every test question this step records the highest similarity between any two of the five retrieved terms (1.0 means identical).

In [22]:
# Finding the highest similarity between two of the five retrieved terms, per question.
highest_values = []
for question, correct_name, correct_identifier in known_questions:
    found = search_terms(question)
    five_embeddings = term_embeddings[found.index.tolist()]
    pairwise = five_embeddings @ five_embeddings.T
    pairwise.fill_diagonal_(-1.0)                                 # ignore each term compared with itself
    highest_values.append(pairwise.max().item())
    print(str(round(highest_values[-1], 3)) + " | " + question)
report("Mean of the highest similarity between two retrieved terms: " + str(round(float(numpy.mean(highest_values)), 3)))

0.761 | How do cells digest and recycle their own components?
0.698 | How do cells repair damaged genetic material?
0.864 | What is the process of programmed cell death?
0.924 | How do cells respond to oxidative stress?
0.851 | How is insulin released from pancreatic cells?
0.851 | How do neurons pass signals across a synapse?
0.82 | chromatin remodelling
0.98 | How does the body defend itself against bacteria?
Mean of the highest similarity between two retrieved terms: 0.844


## Step 10. Gene list test with an enrichment analysis as the answer key

This stage will take a list of human genes, ask which Gene Ontology terms are **over-represented** in it (the answer key), and finally, ask the retrieval pipeline the same question from the gene symbols and compare.

**The test** (over-representation analysis): for each term, the hypergeometric test gives the probability of finding at least this many genes of the list among the genes of the term if the list had been drawn at random from all annotated human genes. The probabilities are corrected for testing many terms with the **Benjamini-Hochberg** method (false discovery rate). A term is significant if the corrected value is below 0.05.

**Choices (all are simplifications):**
1. Annotations are propagated upward (Step 3).
2. Only the `biological_process` branch is tested, with terms of 5 to 1,500 genes (very broad terms are always enriched and say little). Terms are tested only if at least 2 list genes belong to them.
3. The background is every human gene with at least one annotation.
4. **The list is planted:** 30 genes come from the genes of one chosen term (`autophagy`) and 10 random genes are added as noise. This is **circular** (the list was built from the same annotation file that the test uses), so it shows that the code works but flatters the enrichment analysis.
5. **Seeds 0, 1 and 42** give three different random lists. The spread between them only reflects which genes were drawn; it is not the full uncertainty.

**Controls:** a random list of 40 genes should give no significant term, and the enrichment test is checked once against an exact counting calculation.

In [23]:
# Setting the enrichment test and the agreement measure.
ENRICHMENT_NAMESPACE = "biological_process"
MINIMUM_TERM_SIZE = 5
MAXIMUM_TERM_SIZE = 1500
MINIMUM_OVERLAP = 2
ANSWER_KEY_SIZE = 10
SEED_TERM_NAME = "autophagy"
PLANTED_GENE_COUNT = 30
NOISE_GENE_COUNT = 10


# HELPER FUNCTION
def adjust_probability_values(probability_values):
    # Benjamini-Hochberg correction, written out step by step.
    probability_values = numpy.asarray(probability_values, dtype=float)
    count = len(probability_values)
    order = numpy.argsort(probability_values)
    scaled = probability_values[order] * count / numpy.arange(1, count + 1)
    scaled = numpy.minimum.accumulate(scaled[::-1])[::-1]          # make the sequence non-decreasing
    adjusted = numpy.empty(count)
    adjusted[order] = numpy.minimum(scaled, 1.0)
    return adjusted


# HELPER FUNCTION
def run_enrichment(gene_list):
    genes_in_background = [gene for gene in gene_list if gene in gene_to_terms]
    list_size = len(genes_in_background)
    background_size = len(gene_to_terms)
    overlap_counts = collections.Counter()
    for gene in genes_in_background:
        for term_identifier in gene_to_terms[gene]:
            overlap_counts[term_identifier] += 1
    rows = []
    for term_identifier, overlap in overlap_counts.items():
        term = term_by_identifier.get(term_identifier)
        term_size = len(term_to_genes[term_identifier])
        if term is None or term["namespace"] != ENRICHMENT_NAMESPACE:
            continue
        if overlap < MINIMUM_OVERLAP or term_size < MINIMUM_TERM_SIZE or term_size > MAXIMUM_TERM_SIZE:
            continue
        probability_value = scipy.stats.hypergeom.sf(overlap - 1, background_size, term_size, list_size)
        rows.append({"identifier": term_identifier, "name": term["name"], "genes_in_list": overlap,
                     "genes_in_term": term_size, "probability_value": probability_value})
    table = pandas.DataFrame(rows)
    if len(table) == 0:
        return table
    table["adjusted_probability_value"] = adjust_probability_values(table["probability_value"])
    return table.sort_values("adjusted_probability_value").reset_index(drop=True)


# HELPER FUNCTION
def agreement_with_answer_key(identifiers, answer_key_identifiers):
    # Returns (how many identifiers, exact matches, related matches) against the answer key.
    exact = [identifier for identifier in identifiers if identifier in answer_key_identifiers]
    related = [identifier for identifier in identifiers
               if any(terms_are_related(identifier, key) for key in answer_key_identifiers)]
    return len(identifiers), len(exact), len(related)


seed_term_identifier = find_identifier_by_name(SEED_TERM_NAME)
report("Seed term: " + str(seed_term_identifier) + " | " + SEED_TERM_NAME + " | genes annotated to it (after propagation): " + str(len(term_to_genes[seed_term_identifier])))
if not (MINIMUM_TERM_SIZE <= len(term_to_genes[seed_term_identifier]) <= MAXIMUM_TERM_SIZE):
    print("WARNING: the seed term is outside the size limits and will not be tested. Choose another SEED_TERM_NAME.")

Seed term: GO:0006914 | autophagy | genes annotated to it (after propagation): 268


In [24]:
# checking on  the enrichment test once against an exact counting calculation (a control).
control_genes = sorted(term_to_genes[seed_term_identifier])[:PLANTED_GENE_COUNT]
control_result = run_enrichment(control_genes)
control_row = control_result[control_result["identifier"] == seed_term_identifier].iloc[0]
background_size = len(gene_to_terms)
term_size = int(control_row["genes_in_term"])
list_size = len([gene for gene in control_genes if gene in gene_to_terms])
overlap = int(control_row["genes_in_list"])
exact_probability = sum(math.comb(term_size, k) * math.comb(background_size - term_size, list_size - k)
                        for k in range(overlap, min(term_size, list_size) + 1)) / math.comb(background_size, list_size)
report("Control: enrichment probability value " + format(control_row["probability_value"], ".3e")
       + " | exact counting " + format(exact_probability, ".3e")
       + " | agree: " + str(math.isclose(control_row["probability_value"], exact_probability, rel_tol=1e-6)))

Control: enrichment probability value 2.807e-66 | exact counting 2.807e-66 | agree: True


In [25]:
# for each seed, plants a gene list, computes the answer key, runs a random control list, and asks the
# retrieval pipeline (and every working backend) the same question built from the gene symbols.
seed_results = []
seed_term_genes = sorted(term_to_genes[seed_term_identifier])
other_genes = sorted(set(gene_to_terms) - set(seed_term_genes))
all_gene_symbols = sorted(gene_to_terms)

for seed_value in RANDOM_SEEDS:
    random_generator = numpy.random.default_rng(seed_value)
    planted_genes = random_generator.choice(seed_term_genes, size=PLANTED_GENE_COUNT, replace=False).tolist()
    noise_genes = random_generator.choice(other_genes, size=NOISE_GENE_COUNT, replace=False).tolist()
    gene_list = sorted(set(planted_genes) | set(noise_genes))

    enrichment_result = run_enrichment(gene_list)
    significant = enrichment_result[enrichment_result["adjusted_probability_value"] < 0.05]
    answer_key = significant.head(ANSWER_KEY_SIZE)
    answer_key_identifiers = answer_key["identifier"].tolist()
    seed_term_rank = (enrichment_result["identifier"].tolist().index(seed_term_identifier) + 1
                      if seed_term_identifier in enrichment_result["identifier"].tolist() else None)

    random_list = random_generator.choice(all_gene_symbols, size=40, replace=False).tolist()
    random_result = run_enrichment(random_list)
    random_significant = int((random_result["adjusted_probability_value"] < 0.05).sum()) if len(random_result) > 0 else 0

    print()
    print("SEED " + str(seed_value) + " | gene list: " + ", ".join(gene_list))
    report("Seed " + str(seed_value) + ": significant terms " + str(len(significant)) + " | rank of the seed term " + str(seed_term_rank) + " | significant terms in a random control list of 40 genes: " + str(random_significant))
    for position, (_, row) in enumerate(answer_key.iterrows(), start=1):
        print("  answer key " + str(position) + ". " + row["identifier"] + " | " + row["name"] + " | genes " + str(row["genes_in_list"]) + " of " + str(row["genes_in_term"]) + " | adjusted probability value " + format(row["adjusted_probability_value"], ".2e"))

    gene_question = "Which biological processes do these human genes have in common? Genes: " + ", ".join(gene_list)
    found_for_genes = search_terms(gene_question, how_many=HOW_MANY_TERMS, namespace=ENRICHMENT_NAMESPACE)
    retrieved_identifiers = found_for_genes["identifier"].tolist()
    count, exact, related = agreement_with_answer_key(retrieved_identifiers, answer_key_identifiers)
    seed_result = {"seed": seed_value, "retrieval_exact": exact, "retrieval_related": related,
                   "significant_terms": len(significant), "random_control_significant": random_significant}
    report("Seed " + str(seed_value) + " | retrieval only (five terms): exact overlap with the answer key " + str(exact) + ", related overlap " + str(related))

    for backend in BACKENDS_TO_USE:
        for condition in ["with retrieval", "without retrieval"]:
            prompt = build_prompt_with_retrieval(gene_question, found_for_genes) if condition == "with retrieval" else build_prompt_without_retrieval(gene_question)
            answer = generate_answer(prompt, backend)
            checked = check_citations(answer, retrieved_identifiers)
            count, exact, related = agreement_with_answer_key(checked["existing_in_ontology"], answer_key_identifiers)
            report("Seed " + str(seed_value) + " | " + backend + ", " + condition + ": identifiers cited " + str(len(checked["cited"]))
                   + ", not in the ontology " + str(len(checked["not_in_ontology"])) + ", passes the citation check " + str(checked["passes"])
                   + ", exact overlap " + str(exact) + ", related overlap " + str(related))
            seed_result[backend + " " + condition + " exact"] = exact
            seed_result[backend + " " + condition + " related"] = related
    seed_results.append(seed_result)

pandas.DataFrame(seed_results).to_csv(os.path.join(RESULTS_FOLDER, "gene_list_per_seed.csv"), index=False)

SEED 0 | gene list: ABI2, ANXA7, ARL8B, ARMC3, ATG2B, CALM1, DNAJC16, ELAVL1, FAM153B, FBXL4, MAP1S, PEX2, PHB2, RAB1C, RAB2B, RAB39A, RETREG2, SCARB2, SLC15A4, SQSTM1, SYT11, TP53, TRIM32, TSG101, UBA5, ULK1, URS00006D6710_9606, URS00008BB7F2_9606, URS0000D57375_9606, URS0000D5C581_9606, URS0000DE3B7A_9606, URS0002564D32_9606, URS0002594D83_9606, URS0002A360F8_9606, VPS18, VPS28, VTI1A, WDR45B, ZFYVE1, atg5-atg12-tecpr1_human
Seed 0: significant terms 245 | rank of the seed term 2 | significant terms in a random control list of 40 genes: 4
  answer key 1. GO:0061919 | process utilizing autophagic mechanism | genes 30 of 268 | adjusted probability value 3.77e-55
  answer key 2. GO:0006914 | autophagy | genes 30 of 268 | adjusted probability value 3.77e-55
  answer key 3. GO:0016236 | macroautophagy | genes 24 of 229 | adjusted probability value 5.95e-42
  answer key 4. GO:0007033 | vacuole organization | genes 11 of 172 | adjusted probability value 1.63e-15
  answer key 5. GO:1905037 |


SEED 1 | gene list: ARHGAP26, ARL8B, ATG4C, ATP6V0A4, BAG3, CLU, DNAJC16, DTD1, EI24, FTL, GABARAP, HDAC6, MAP1S, MFN2, NIPSNAP2, OPTN, OR2T5, PDZD8, RAB23, RNF19B, SMURF1, STUB1, SYCE1L, TAGAP, TP53, TRIAP1, TRIM32, UBXN6, UFC1, URS00003DEABB_9606, URS00007CC2A3_9606, URS0000EF158F_9606, USP30, UVRAG, VAMP8, VPS13D, VPS36, VPS37C, igha2-iglc1_human, stx17-snap47-vamp7_human
Seed 1: significant terms 325 | rank of the seed term 2 | significant terms in a random control list of 40 genes: 1
  answer key 1. GO:0061919 | process utilizing autophagic mechanism | genes 30 of 268 | adjusted probability value 4.60e-55
  answer key 2. GO:0006914 | autophagy | genes 30 of 268 | adjusted probability value 4.60e-55
  answer key 3. GO:0016236 | macroautophagy | genes 25 of 229 | adjusted probability value 2.47e-44
  answer key 4. GO:0000423 | mitophagy | genes 8 of 54 | adjusted probability value 6.22e-14
  answer key 5. GO:0007033 | vacuole organization | genes 10 of 172 | adjusted probability va


SEED 42 | gene list: ATG4B, ATG4D, ATP6V0A2, CHMP1B, CHMP2B, CHURC1, LAMP2, MFN2, MFSD8, NPC1, OGT, PINK1, PIP4K2A, PIP4K2B, POFUT2, RAB33B, RPLP2, RTP4, RUBCNL, SNX30, SPART, SPPL2B, STUB1, STX4, SYT11, TIMM23, TRAPPC4, UBAC2, UFC1, URS0000B89D28_9606, URS0000EA5560_9606, USP30, VAMP8, VTI1B, WDFY3, WDR45, XPNPEP2, ZNF662, hsa-miR-650, ubxn6-vcp_human
Seed 42: significant terms 296 | rank of the seed term 2 | significant terms in a random control list of 40 genes: 0
  answer key 1. GO:0061919 | process utilizing autophagic mechanism | genes 30 of 268 | adjusted probability value 4.07e-55
  answer key 2. GO:0006914 | autophagy | genes 30 of 268 | adjusted probability value 4.07e-55
  answer key 3. GO:0016236 | macroautophagy | genes 26 of 229 | adjusted probability value 6.67e-47
  answer key 4. GO:0048284 | organelle fusion | genes 11 of 149 | adjusted probability value 3.50e-16
  answer key 5. GO:0000422 | autophagy of mitochondrion | genes 9 of 70 | adjusted probability value 2.25e

In [26]:
# mean and standard deviation over the seeds (sample standard deviation, n minus 1).
seed_table = pandas.DataFrame(seed_results)
for column_name in seed_table.columns:
    if column_name == "seed":
        continue
    values = seed_table[column_name].to_numpy(dtype=float)
    report(column_name + ": mean " + str(round(float(values.mean()), 2)) + ", standard deviation " + str(round(float(values.std(ddof=1)), 2)) + " over seeds " + str(RANDOM_SEEDS))

retrieval_exact: mean 0.0, standard deviation 0.0 over seeds [0, 1, 42]
retrieval_related: mean 0.0, standard deviation 0.0 over seeds [0, 1, 42]
significant_terms: mean 288.67, standard deviation 40.5 over seeds [0, 1, 42]
random_control_significant: mean 1.67, standard deviation 2.08 over seeds [0, 1, 42]
local with retrieval exact: mean 0.0, standard deviation 0.0 over seeds [0, 1, 42]
local with retrieval related: mean 0.0, standard deviation 0.0 over seeds [0, 1, 42]
local without retrieval exact: mean 0.0, standard deviation 0.0 over seeds [0, 1, 42]
local without retrieval related: mean 0.0, standard deviation 0.0 over seeds [0, 1, 42]


## Step 11. The methylation bridge: your own sites as gene lists

**What and why:** until now the gene lists were planted. This step uses the real lists from the methylation project: the sites that the elastic net gave a weight other than zero (seeds 0, 1 and 42, and the sites that had one in all three seeds). It turns each list of sites into a list of genes, runs the same kind of over-representation test as Step 10, checks it with a permutation test, and compares the result with the R package `missMethyl` (`01_missmethyl_check.R`).

**Inputs:** `input/methylation_repo/` (the site lists and `all_sites_in_study.txt`) and `data/site_to_gene_table.csv` (written by `01_missmethyl_check.R` from the Bioconductor annotation of the EPIC array).

**Choices (all are simplifications):**
1. Only sites whose name starts with `cg` are used; the `ch.` sites have no gene annotation.
2. A site belongs to every gene named for it in the annotation table. Sites with no gene name are counted as unmapped and are dropped.
3. **The background** is not every annotated human gene (as in Step 10) but only the genes that have at least one site in the study and a matching symbol in the UniProt annotation. Otherwise genes that the array cannot see would count as "not in the list".
4. **Bias:** genes with many sites on the array are more likely to appear in a list by chance, and the hypergeometric test ignores this. The permutation test addresses it: lists with the same number of random sites from the study are drawn, mapped to genes and tested in the same way, and the observed probability values are compared with those.
5. Gene symbols come from two sources (the array annotation and the UniProt annotation file) and can be spelled differently. Genes without a matching symbol are counted and dropped.

**Expected output:** per list, the counts of sites and genes; the top terms with the genes behind them; the permutation result; and the comparison with `missMethyl`. Files: `results/python_enrichment_<list>.csv` and `results/bridge_summary.csv`.

In [27]:
# Settings of the bridge step, the four lists of sites, the site-to-gene table and the background genes.
METHYLATION_FOLDER = os.path.join("input", "methylation_repo")
SITE_TO_GENE_PATH = os.path.join(DATA_FOLDER, "site_to_gene_table.csv")
ALL_SITES_PATH = os.path.join(METHYLATION_FOLDER, "all_sites_in_study.txt")
PERMUTATIONS = 500                   # random lists drawn for the permutation test
TOP_TERMS = 20                       # terms for which the genes behind them and the permutation values are kept
TERMS_TO_PRINT = 10
BRIDGE_SEED = 0

if not os.path.exists(SITE_TO_GENE_PATH):
    raise FileNotFoundError(SITE_TO_GENE_PATH + " is missing. Run 01_missmethyl_check.R first (Rscript 01_missmethyl_check.R); it writes this file.")

# The four lists of sites (the same rule as in 01_missmethyl_check.R).
sites_by_list = {}
for seed_value in RANDOM_SEEDS:
    seed_table = pandas.read_csv(os.path.join(METHYLATION_FOLDER, "elastic_net_sites_seed_" + str(seed_value) + ".csv"))
    weight_is_not_zero = seed_table["weight_is_not_zero"].astype(str).str.lower() == "true"
    chosen_sites = seed_table.loc[weight_is_not_zero, "site"].tolist()
    cg_sites = [site for site in chosen_sites if site.startswith("cg")]
    sites_by_list["seed_" + str(seed_value)] = cg_sites
    report("Seed " + str(seed_value) + ": weights other than zero " + str(len(chosen_sites)) + " | after removing ch. sites " + str(len(cg_sites)))
sites_by_list["all_seeds"] = sorted(set.intersection(*[set(sites) for sites in sites_by_list.values()]))
report("Sites with a weight other than zero in all seeds: " + str(len(sites_by_list["all_seeds"])))

# The site-to-gene table: one text per site with the gene names separated by semicolons (empty if the site has no gene).
site_gene_table = pandas.read_csv(SITE_TO_GENE_PATH, usecols=["site", "gene_names"], keep_default_na=False)
site_index = pandas.Index(site_gene_table["site"])
gene_name_strings = site_gene_table["gene_names"].to_numpy()
report("Site-to-gene table: " + str(len(site_gene_table)) + " sites, of which with at least one gene name " + str(int((gene_name_strings != "").sum())))


# HELPER FUNCTION
def genes_of_sites(sites):
    # The set of genes named for these sites in the annotation table (duplicates inside a site are removed).
    positions = site_index.get_indexer(sites)
    positions = positions[positions >= 0]
    genes = set()
    for text in gene_name_strings[positions]:
        if text != "":
            genes.update(text.split(";"))
    return genes


# HELPER FUNCTION
def count_mapping(sites):
    # How many sites are missing from the table and how many have no gene name.
    positions = site_index.get_indexer(sites)
    in_table = positions[positions >= 0]
    without_gene = int((gene_name_strings[in_table] == "").sum())
    return {"sites": len(sites), "not_in_table": int((positions < 0).sum()), "without_gene": without_gene}


# The background: the genes named for the cg sites of the study that also have an annotated symbol.
if os.path.exists(ALL_SITES_PATH):
    with open(ALL_SITES_PATH, encoding="utf-8") as file:
        study_sites = [line.strip() for line in file if line.strip() != ""]
    background_source = "all sites of the study"
else:
    study_sites = site_gene_table["site"].tolist()
    background_source = "all sites of the array (all_sites_in_study.txt was not found)"
background_sites = numpy.array([site for site in study_sites if site.startswith("cg")])
array_genes = genes_of_sites(background_sites)
background_genes = sorted(gene for gene in array_genes if gene in gene_to_terms)
report("Background: " + background_source + " | cg sites " + str(len(background_sites)) + " | genes named for them " + str(len(array_genes))
       + " | of which with a matching annotated symbol (the background genes) " + str(len(background_genes)))
print("Examples of gene names without a matching annotated symbol:", sorted(array_genes - set(gene_to_terms))[:10])

Seed 0: weights other than zero 126 | after removing ch. sites 126
Seed 1: weights other than zero 755 | after removing ch. sites 753
Seed 42: weights other than zero 186 | after removing ch. sites 186
Sites with a weight other than zero in all seeds: 36


Site-to-gene table: 865859 sites, of which with at least one gene name 616598


Background: all sites of the study | cg sites 765569 | genes named for them 25670 | of which with a matching annotated symbol (the background genes) 16686
Examples of gene names without a matching annotated symbol: ['A1BG-AS1', 'A2BP1', 'A2LD1', 'A2M-AS1', 'A2MP1', 'AA06', 'AAA1', 'AACSL', 'AACSP1', 'AADACL2-AS1']


In [28]:
# The enrichment test inside the restricted background (the same test as run_enrichment in Step 10, but fast enough for many random lists).
# HELPER FUNCTION
def prepare_background(genes):
    # Counts the size of every term inside this background and keeps the terms that Step 10 would test.
    size_of_term = collections.Counter()
    for gene in genes:
        for term_identifier in gene_to_terms[gene]:
            size_of_term[term_identifier] += 1
    tested_terms = [identifier for identifier, size in size_of_term.items()
                    if identifier in term_by_identifier
                    and term_by_identifier[identifier]["namespace"] == ENRICHMENT_NAMESPACE
                    and MINIMUM_TERM_SIZE <= size <= MAXIMUM_TERM_SIZE]
    position_of_term = {identifier: position for position, identifier in enumerate(tested_terms)}
    term_positions_of_gene = {}
    for gene in genes:
        term_positions_of_gene[gene] = numpy.array(
            [position_of_term[identifier] for identifier in gene_to_terms[gene] if identifier in position_of_term], dtype=int)
    return {"genes": set(genes), "terms": tested_terms, "position_of_term": position_of_term,
            "sizes": numpy.array([size_of_term[identifier] for identifier in tested_terms]),
            "term_positions_of_gene": term_positions_of_gene}


# HELPER FUNCTION
def probabilities_in_background(gene_list, prepared):
    # One probability value per tested term (1.0 where fewer than MINIMUM_OVERLAP genes of the list belong to the term), the overlaps and the list size.
    genes = [gene for gene in gene_list if gene in prepared["genes"]]
    term_count = len(prepared["terms"])
    probabilities = numpy.ones(term_count)
    overlaps = numpy.zeros(term_count, dtype=int)
    if len(genes) == 0:
        return probabilities, overlaps, 0
    overlaps = numpy.bincount(numpy.concatenate([prepared["term_positions_of_gene"][gene] for gene in genes]), minlength=term_count)
    tested = overlaps >= MINIMUM_OVERLAP
    # Many terms share the same pair (overlap, term size), so each pair is calculated once and copied back (this makes the test about twice as fast).
    pair_keys = overlaps[tested] * 100000 + prepared["sizes"][tested]
    unique_keys, inverse = numpy.unique(pair_keys, return_inverse=True)
    unique_probabilities = scipy.stats.hypergeom.sf(unique_keys // 100000 - 1, len(prepared["genes"]), unique_keys % 100000, len(genes))
    probabilities[tested] = unique_probabilities[inverse]
    return probabilities, overlaps, len(genes)


# HELPER FUNCTION
def enrichment_table(gene_list, prepared):
    # The result table in the same layout as run_enrichment: Benjamini-Hochberg over the tested terms, lowest probability value first.
    probabilities, overlaps, list_size = probabilities_in_background(gene_list, prepared)
    tested_positions = numpy.flatnonzero(overlaps >= MINIMUM_OVERLAP)
    table = pandas.DataFrame({
        "identifier": [prepared["terms"][position] for position in tested_positions],
        "name": [term_by_identifier[prepared["terms"][position]]["name"] for position in tested_positions],
        "genes_in_list": overlaps[tested_positions],
        "genes_in_term": prepared["sizes"][tested_positions],
        "probability_value": probabilities[tested_positions],
    })
    table["adjusted_probability_value"] = adjust_probability_values(table["probability_value"]) if len(table) > 0 else []
    table = table.sort_values(["probability_value", "identifier"]).reset_index(drop=True)
    return table, list_size


prepared_background = prepare_background(background_genes)
report("Terms tested in the restricted background (biological process, " + str(MINIMUM_TERM_SIZE) + " to " + str(MAXIMUM_TERM_SIZE) + " genes inside the background): " + str(len(prepared_background["terms"])))

Terms tested in the restricted background (biological process, 5 to 1500 genes inside the background): 7729


In [29]:
# The enrichment test for each of the four lists, with the genes behind the top terms.
bridge_results = {}
for list_name, list_sites in sites_by_list.items():
    mapping = count_mapping(list_sites)
    list_genes = genes_of_sites(list_sites)
    genes_in_background = sorted(gene for gene in list_genes if gene in prepared_background["genes"])
    table, list_size = enrichment_table(genes_in_background, prepared_background)

    table["genes_in_list_names"] = ""                          # the genes of the list that belong to the term, for the top terms only
    for row_number in range(min(TOP_TERMS, len(table))):
        term_identifier = table.loc[row_number, "identifier"]
        table.loc[row_number, "genes_in_list_names"] = ";".join(gene for gene in genes_in_background if term_identifier in gene_to_terms[gene])
    significant = int((table["adjusted_probability_value"] < 0.05).sum()) if len(table) > 0 else 0

    report("Methylation list " + list_name + ": sites " + str(mapping["sites"]) + " | not in the annotation table " + str(mapping["not_in_table"])
           + " | without a gene name " + str(mapping["without_gene"]) + " | genes named " + str(len(list_genes))
           + " | genes with a matching annotated symbol " + str(len(genes_in_background)) + " | terms tested " + str(len(table))
           + " | significant terms (adjusted below 0.05) " + str(significant))
    for position, (_, row) in enumerate(table.head(TERMS_TO_PRINT).iterrows(), start=1):
        print("  " + str(position) + ". " + row["identifier"] + " | " + row["name"] + " | genes " + str(row["genes_in_list"]) + " of " + str(row["genes_in_term"])
              + " | probability value " + format(row["probability_value"], ".2e") + " | adjusted " + format(row["adjusted_probability_value"], ".2e")
              + " | " + row["genes_in_list_names"])
    table.to_csv(os.path.join(RESULTS_FOLDER, "python_enrichment_" + list_name + ".csv"), index=False)
    bridge_results[list_name] = {"table": table, "genes": genes_in_background, "sites": list_sites, "significant": significant,
                                 "permutation_smallest": numpy.nan}

Methylation list seed_0: sites 126 | not in the annotation table 0 | without a gene name 31 | genes named 98 | genes with a matching annotated symbol 75 | terms tested 566 | significant terms (adjusted below 0.05) 7
  1. GO:0048839 | inner ear development | genes 5 of 95 | probability value 6.78e-05 | adjusted 2.40e-02 | ATP8A2;CALB1;ROR1;SOBP;ZIC1
  2. GO:0043583 | ear development | genes 5 of 105 | probability value 1.09e-04 | adjusted 2.40e-02 | ATP8A2;CALB1;ROR1;SOBP;ZIC1
  3. GO:0032496 | response to lipopolysaccharide | genes 7 of 250 | probability value 1.30e-04 | adjusted 2.40e-02 | CASP1;IL13;LOXL1;PENK;SELP;SOD2;TAB2
  4. GO:0002237 | response to molecule of bacterial origin | genes 7 of 261 | probability value 1.70e-04 | adjusted 2.40e-02 | CASP1;IL13;LOXL1;PENK;SELP;SOD2;TAB2
  5. GO:0009617 | response to bacterium | genes 9 of 473 | probability value 2.62e-04 | adjusted 2.96e-02 | AQP1;CASP1;CHGA;IL13;LOXL1;PENK;SELP;SOD2;TAB2
  6. GO:0030952 | establishment or maintenance

Methylation list seed_42: sites 186 | not in the annotation table 0 | without a gene name 59 | genes named 139 | genes with a matching annotated symbol 118 | terms tested 839 | significant terms (adjusted below 0.05) 13
  1. GO:0007156 | homophilic cell-cell adhesion | genes 17 of 166 | probability value 2.34e-15 | adjusted 1.97e-12 | CDH7;PCDHA1;PCDHA10;PCDHA11;PCDHA12;PCDHA13;PCDHA2;PCDHA3;PCDHA4;PCDHA5;PCDHA6;PCDHA7;PCDHA8;PCDHA9;PCDHAC1;PCDHGA1;PCDHGA2
  2. GO:0098609 | cell-cell adhesion | genes 22 of 490 | probability value 3.79e-12 | adjusted 1.59e-09 | ADGRV1;CDH7;CX3CL1;DLG1;PCDHA1;PCDHA10;PCDHA11;PCDHA12;PCDHA13;PCDHA2;PCDHA3;PCDHA4;PCDHA5;PCDHA6;PCDHA7;PCDHA8;PCDHA9;PCDHAC1;PCDHGA1;PCDHGA2;SPARCL1;TNF
  3. GO:0007155 | cell adhesion | genes 25 of 755 | probability value 7.99e-11 | adjusted 2.24e-08 | ADGRV1;CDH7;CX3CL1;DLG1;PCDHA1;PCDHA10;PCDHA11;PCDHA12;PCDHA13;PCDHA2;PCDHA3;PCDHA4;PCDHA5;PCDHA6;PCDHA7;PCDHA8;PCDHA9;PCDHAC1;PCDHGA1;PCDHGA2;PLXNC1;PXN;SPARCL1;SSPN;TNF
  4. G

In [30]:
# The permutation test: the same number of random sites, drawn from the background sites, mapped to genes and tested in the same way.
# Two numbers per list: (1) for each of the top terms, the share of random lists with an equal or smaller probability value for that term;
# (2) the share of random lists whose SMALLEST probability value over all terms is as small as the observed smallest one (this one needs no correction for many terms).
random_generator = numpy.random.default_rng(BRIDGE_SEED)
for list_name, stored in bridge_results.items():
    table = stored["table"]
    if len(table) == 0:
        report("Permutation test, list " + list_name + ": no term was tested, skipped.")
        continue
    top_count = min(TOP_TERMS, len(table))
    top_positions = numpy.array([prepared_background["position_of_term"][identifier] for identifier in table["identifier"].head(top_count)])
    observed_top = table["probability_value"].head(top_count).to_numpy()
    observed_smallest = observed_top[0]                        # the table is sorted, so the first value is the smallest
    count_top = numpy.zeros(top_count)
    count_smallest = 0
    for _ in range(PERMUTATIONS):
        random_sites = random_generator.choice(background_sites, size=len(stored["sites"]), replace=False)
        random_genes = genes_of_sites(random_sites)
        random_probabilities = probabilities_in_background(random_genes, prepared_background)[0]
        count_top += random_probabilities[top_positions] <= observed_top
        count_smallest += int(random_probabilities.min() <= observed_smallest)
    table["permutation_probability_value"] = numpy.nan
    table.loc[table.index[:top_count], "permutation_probability_value"] = (count_top + 1) / (PERMUTATIONS + 1)
    stored["permutation_smallest"] = (count_smallest + 1) / (PERMUTATIONS + 1)
    report("Permutation test, list " + list_name + " (" + str(PERMUTATIONS) + " random lists of " + str(len(stored["sites"])) + " sites): smallest probability value observed "
           + format(observed_smallest, ".2e") + " | share of random lists with an equal or smaller smallest value " + format(stored["permutation_smallest"], ".3f")
           + " | top term: " + table.loc[0, "name"] + " (permutation value " + format(table.loc[0, "permutation_probability_value"], ".3f") + ")")
    table.to_csv(os.path.join(RESULTS_FOLDER, "python_enrichment_" + list_name + ".csv"), index=False)

Permutation test, list seed_0 (500 random lists of 126 sites): smallest probability value observed 6.78e-05 | share of random lists with an equal or smaller smallest value 0.232 | top term: inner ear development (permutation value 0.002)


Permutation test, list seed_1 (500 random lists of 753 sites): smallest probability value observed 2.20e-09 | share of random lists with an equal or smaller smallest value 0.160 | top term: homophilic cell-cell adhesion (permutation value 0.124)


Permutation test, list seed_42 (500 random lists of 186 sites): smallest probability value observed 2.34e-15 | share of random lists with an equal or smaller smallest value 0.026 | top term: homophilic cell-cell adhesion (permutation value 0.020)


Permutation test, list all_seeds (500 random lists of 36 sites): smallest probability value observed 3.11e-04 | share of random lists with an equal or smaller smallest value 0.399 | top term: stem cell proliferation (permutation value 0.002)


In [31]:
# The comparison with the missMethyl result 
# (results/missmethyl_<list>.csv, written by 01_missmethyl_check.R) and the summary file.
summary_rows = []
for list_name, stored in bridge_results.items():
    table = stored["table"]
    row = {"list": list_name, "sites": len(stored["sites"]), "genes_with_matching_symbol": len(stored["genes"]),
           "python_significant_terms": stored["significant"],
           "python_smallest_probability_value": float(table["probability_value"].min()) if len(table) > 0 else numpy.nan,
           "permutation_share_smallest": stored["permutation_smallest"],
           "missmethyl_significant_terms": numpy.nan, "top_ten_shared": numpy.nan, "rank_correlation": numpy.nan, "terms_compared": numpy.nan}
    missmethyl_path = os.path.join(RESULTS_FOLDER, "missmethyl_" + list_name + ".csv")
    if os.path.exists(missmethyl_path) and len(table) > 0:
        missmethyl_table = pandas.read_csv(missmethyl_path, index_col=0)          # the first column holds the term identifiers
        missmethyl_top = missmethyl_table.head(ANSWER_KEY_SIZE).index.tolist()    # the file is ordered by probability value
        python_top = table["identifier"].head(ANSWER_KEY_SIZE).tolist()
        joined = table.merge(missmethyl_table[["P.DE", "FDR"]], left_on="identifier", right_index=True)
        correlation = scipy.stats.spearmanr(joined["probability_value"], joined["P.DE"]).statistic if len(joined) > 2 else numpy.nan
        row["missmethyl_significant_terms"] = int((missmethyl_table["FDR"] < 0.05).sum())
        row["top_ten_shared"] = len(set(missmethyl_top) & set(python_top))
        row["rank_correlation"] = round(float(correlation), 3)
        row["terms_compared"] = len(joined)
        report("Comparison with missMethyl, list " + list_name + ": significant terms " + str(stored["significant"]) + " here and " + str(row["missmethyl_significant_terms"])
               + " there | terms shared among the ten lowest " + str(row["top_ten_shared"]) + " of " + str(ANSWER_KEY_SIZE)
               + " | rank correlation of the probability values (Spearman) " + str(row["rank_correlation"]) + " over " + str(len(joined)) + " terms in both tables")
    else:
        report("Comparison with missMethyl, list " + list_name + ": " + missmethyl_path + " was not found, skipped.")
    summary_rows.append(row)

pandas.DataFrame(summary_rows).to_csv(os.path.join(RESULTS_FOLDER, "bridge_summary.csv"), index=False)
print("Saved", os.path.join(RESULTS_FOLDER, "bridge_summary.csv"))

Comparison with missMethyl, list seed_0: significant terms 7 here and 0 there | terms shared among the ten lowest 8 of 10 | rank correlation of the probability values (Spearman) 0.825 over 566 terms in both tables
Comparison with missMethyl, list seed_1: significant terms 25 here and 0 there | terms shared among the ten lowest 0 of 10 | rank correlation of the probability values (Spearman) 0.764 over 2195 terms in both tables
Comparison with missMethyl, list seed_42: significant terms 13 here and 0 there | terms shared among the ten lowest 5 of 10 | rank correlation of the probability values (Spearman) 0.828 over 839 terms in both tables
Comparison with missMethyl, list all_seeds: significant terms 3 here and 0 there | terms shared among the ten lowest 1 of 10 | rank correlation of the probability values (Spearman) 0.756 over 68 terms in both tables
Saved results\bridge_summary.csv


## Step 12. Conclusions

All numbers below are copied from the outputs of this saved notebook. Only the **local backend** was run (`Qwen/Qwen2.5-0.5B-Instruct`, a small model that runs on the processor); the Claude backend was not run, so nothing here says how a stronger model behaves. Ontology release 2026-07-26; the human annotation file was generated by UniProt on 2026-07-28. Step 11 uses real lists of methylation sites; the retrieval pipeline and the language models were **not** run on them.

### Retrieval (Step 8.1)

- 8 of the 10 test questions had a known correct term. Two questions were skipped because the exact name was not found in the ontology file: `aging` (question "How do cells get older?") and `DNA methylation` (question "How is a methyl group added to genetic material?"). These are the two questions closest to the methylation project, so they were not tested. Why the names were not found was not tested here.
- Embedding search: the correct term was first in 3 of 8 questions, among the five retrieved terms in 5 of 8, and a related term (the same term, an ancestor or a descendant) was among the five in 7 of 8.
- Naive word search (the baseline): the correct term was among the five in 2 of 8, and a related term in 5 of 8.
- Chance level for the correct term being among five random terms: 0.00013 per question (5 of 38,092 terms).
- The correct term was not among the five for 3 questions: `autophagy` (top result "digestion"), `insulin secretion` (top result "insulin catabolic process") and `chemical synaptic transmission` (top result "synaptic signaling").
- **Reading:** embedding search found more correct terms than word search in these 8 questions, and far more than chance. A difference of three questions (5 against 2) is too small to call reliable.

### Generation with the local model (Step 8.2)

| Condition | Answers | Identifiers cited | Existing in the ontology | Inside the retrieved five | Answers passing the strict check | Answers with a relevant term |
|---|---|---|---|---|---|---|
| with retrieval | 8 | 5 | 5 | 5 | 1 | 1 |
| without retrieval | 8 | 0 | 0 | 0 | 0 | 0 |

- With retrieval, every identifier that was cited was real and was among the retrieved five. But the strict check passes only when an answer cites at least one identifier, so 7 of 8 answers failed it by citing **no identifier at all**, even though the instruction asked for citations.
- Without retrieval, the model cited **no identifier at all**. So this test could **not** show whether retrieval prevents invented identifiers: the small model did not write any. This question stays open for a stronger model.
- Two single answers from the demonstration in Step 7 (not a measure): the answer without retrieval switched into Chinese halfway, and the answer with retrieval ended with a sentence that went beyond the five terms ("transcriptional regulation, epigenetic modifications ..."). The citation check does not catch statements that are not tied to an identifier.

### Near-duplicate terms (Step 9)

The mean of the highest similarity between two of the five retrieved terms was 0.844 (per question from 0.698 to 0.98). The five retrieved terms are therefore often close to each other in meaning, which gives less variety than five separate terms would.

### Gene list test (Step 10)

| Seed | Significant terms | Rank of the seed term `autophagy` | Significant terms in a random list of 40 genes | Retrieval only: overlap with the answer key (exact, related) | Local with retrieval: identifiers cited | Local without retrieval: identifiers cited |
|---|---|---|---|---|---|---|
| 0 | 245 | 1 | 4 | 0, 0 | 5 | 0 |
| 1 | 325 | 2 | 1 | 0, 0 | 3 | 0 |
| 42 | 296 | 1 | 0 | 0, 0 | 3 | 0 |
| mean (standard deviation) | 288.67 (40.5) | | 1.67 (2.08) | 0 (0) | | |

- **The enrichment analysis recovered the planted term.** The list was built from the genes of `autophagy` plus 10 random genes, so this is **circular**: it shows that the code works, not that the method works on a real gene list. For seed 1 the seed term is rank 2 only because it ties with `process utilizing autophagic mechanism`, which has the same counts (30 of 268 genes) and the identical adjusted probability value.
- **The code check passed:** for one term, the enrichment probability (2.807e-66) and an exact counting calculation agree. This checks the arithmetic of the probability, not the correction for many tests.
- **The random control did not behave as the brief expected.** A random list of 40 genes should give no significant term, but gave 4, 1 and 0. The reasons were not tested here (possibilities: terms that overlap strongly, entries that are not genes, only three random lists).
- **Retrieval did not reproduce the answer key.** The five terms retrieved for the gene question matched none of the top ten terms of the answer key in any seed, neither exactly nor as ancestor or descendant. The local model with retrieval cited identifiers (5, 3 and 3), all real and among the retrieved five, but none was in the answer key. Without retrieval it cited none. Why retrieval failed was not tested here (possibilities: the embedding model reads gene symbols as ordinary text; the question may be longer than the model's limit of 256 word pieces).
- **Data note:** the gene lists contain entries that are not gene symbols: RNAcentral identifiers starting with `URS` (8, 3 and 2 in the three lists), protein complexes ending in `_human` (1, 2 and 1), and one microRNA (`hsa-miR-650`, seed 42). The UniProt-based annotation file includes such entries, and they are part of the background of 38,810 entries. The effect on the results was not tested here.

### Methylation bridge (Step 11)

The lists are the sites that the elastic net of the methylation project gave a weight other than zero: 126 (seed 0), 753 (seed 1) and 186 (seed 42) `cg` sites after removing `ch.` sites, and 36 sites that had a weight in all three seeds. Sites were turned into genes with the Bioconductor annotation table of the EPIC array (865,859 sites, 616,598 of them with at least one gene name). The background was the 765,569 `cg` sites of the study: they are named for 25,670 genes, of which 16,686 have a matching symbol in the UniProt annotation (the background genes). The lists name 98 (seed 0), 525 (seed 1) and 139 (seed 42) genes; 75, 434 and 118 of them have a matching symbol. 31, 232 and 59 sites have no gene name. 7,729 biological-process terms were tested in this background.

| List | Sites | Significant terms here (adjusted below 0.05) | Significant terms in missMethyl | Smallest value in this test | Share of 500 random lists with an equal or smaller smallest value | Top term | Top-10 shared with missMethyl | Rank correlation (Spearman) |
|---|---|---|---|---|---|---|---|---|
| seed_0 | 126 | 7 | 0 | 6.78e-05 | 0.232 | inner ear development | 8 | 0.825 |
| seed_1 | 753 | 25 | 0 | 2.20e-09 | 0.160 | homophilic cell-cell adhesion | 0 | 0.764 |
| seed_42 | 186 | 13 | 0 | 2.34e-15 | 0.026 | homophilic cell-cell adhesion | 5 | 0.828 |
| all_seeds | 36 | 3 | 0 | 3.11e-04 | 0.399 | stem cell proliferation | 1 | 0.756 |

- **No list shows reliable enrichment.** The hypergeometric test of this notebook gave 7, 25, 13 and 3 significant terms, but `missMethyl` (which corrects for the number of array sites per gene) gave none in any of the four lists.
- **The permutation test agrees with `missMethyl`.** For seeds 0 and 1 and for `all_seeds`, random lists of the same number of sites give a best term as extreme as the observed one in 23%, 16% and 40% of the 500 draws. Seed 42 gave 2.6%: below 0.05, but not below 0.0125 (0.05 divided by four lists), and the four lists share sites, so they are not independent tests. At most this is a hint.
- **A per-term permutation value of 0.002 is not a result by itself.** It was found for the top terms of seed 0 (`inner ear development`) and `all_seeds` (`stem cell proliferation`), and 0.002 is the smallest value that 500 permutations can give. These terms were picked as the lowest of about 7,700 after looking, and the check above already accounts for that.
- **The raw probability values are too small, and the gene names show why.** For seed 42 the value 2.34e-15 looks very strong, yet 2.6% of random site lists give a value as small. For seed 1 the top term `homophilic cell-cell adhesion` has a raw value of 2.20e-09 but a permutation value of 0.124. 16 of the 17 genes behind the seed 42 top term are protocadherin genes (`PCDHA1` to `PCDHA13`, `PCDHAC1`, `PCDHGA1`, `PCDHGA2`); in seed 1, 14 of the 21 are `PCDHGA1` to `PCDHGA9` and `PCDHGB1` to `PCDHGB5`. These genes lie together in gene clusters, and a site inside a cluster is named for many genes at once, so a few sites are counted as many genes. This was not checked in the site-to-gene table.
- **The significant terms are not independent.** In seed 0 the two lowest terms (`inner ear development`, `ear development`) share the same 5 genes (`ATP8A2`, `CALB1`, `ROR1`, `SOBP`, `ZIC1`), and the four terms about response to bacteria share the same 7 genes. In seed 1 the `GDP metabolic process` and `GMP metabolic process` terms rest on 4 and 6 genes, including `DLG1`, `DLG2`, `GUK1` and `TJP2`. The full tables are in `results/python_enrichment_<list>.csv` and `results/missmethyl_<list>.csv`.
- **The two analyses rank terms alike but are not calibrated alike.** Rank correlations are 0.76 to 0.83 over the terms in both tables, but the overlap of the top ten varies from 0 to 8 of 10. They use different gene-to-term tables (UniProt file here, `org.Hs.eg.db` and `GO.db` there), so exact agreement was not expected.
- **Symbols lost:** 25,670 gene names in the background became 16,686 matched genes. In the lists, 98, 525 and 139 named genes became 75, 434 and 118. Some of the unmatched names are antisense or pseudogene entries and some look like older symbols (for example `A2BP1`). They were dropped. Mapping through Entrez identifiers instead of symbols was not tested.

### What these outputs support, and what they do not

- They support: embedding search finds the correct term or a related term more often than word search or chance in this small question set; when the small local model cites, it cites real terms from the retrieved list; the enrichment code recovers a planted term; the methylation lists from the elastic net show no reliable GO enrichment, by two analyses (`missMethyl` and a permutation test).
- They do **not** support: that retrieval prevents invented identifiers (none were written without retrieval); that retrieval helps a stronger model (not run); that retrieval agrees with an enrichment analysis on a real gene list (the planted list had zero overlap, and the real lists have no significant terms to use as an answer key); that these sites are unrelated to biology, since a list chosen to predict age is not a list of sites known to change with age.

In [32]:
# saving every reported line to results/summary.txt for the README.
with open(os.path.join(RESULTS_FOLDER, "summary.txt"), "w", encoding="utf-8") as file:
    file.write("\n".join(summary_lines) + "\n")
print("Saved", len(summary_lines), "lines to", os.path.join(RESULTS_FOLDER, "summary.txt"))

Saved 76 lines to results\summary.txt


### Limits

- Only 8 test questions with one correct term each, and 3 seeds. The standard deviation over seeds shows only which genes were drawn, not the full uncertainty.
- One small model (about half a billion numbers), run once, with no tuning of the prompt.
- A planted gene list built from the same annotation file the test uses.
- Annotation terms were propagated to ancestors, only `biological_process` was tested, and the background of Step 10 includes entries that are not genes.
- **Methylation bridge (Step 11):**
  - The lists are the sites that the elastic net gave a weight other than zero (126, 753, 186 and 36 sites after removing `ch.` sites, see the output). They are small, and the elastic net of seed 1 did not converge. A list chosen by a model that predicts age is not a list of sites that are known to change with age.
  - Sites are turned into genes with the Bioconductor annotation of the EPIC array version 1 (genome build hg19). A site can belong to several genes, and sites without a gene name are dropped (the numbers are in the output).
  - Gene symbols come from two sources (array annotation and the UniProt file); symbols without a match are dropped and counted.
  - The permutation test only addresses the number of sites per gene. It does not address other structure, for example sites that lie close together and behave alike. Its smallest possible value is 1 / (number of permutations + 1).
  - `missMethyl` uses its own gene-to-term tables (`org.Hs.eg.db` and `GO.db`), not the UniProt file used here, so the two results are not expected to agree exactly. Its column `genes_in_list` can hold fewer genes than its column `DE`; the reason was not tested.
  - The retrieval pipeline and the language models were not run on the methylation lists.
- For learning, not for scientific or clinical use.

### Next steps

1. Run the Claude backend (the answers are cached, so a repeated run costs nothing) and compare it with the local model.
2. Find the exact names in this ontology release for the two skipped questions, and rerun Step 8.
3. Ask the retrieval pipeline and the language models the same question for the genes of the methylation lists, and compare with the enrichment result. This is only meaningful if the enrichment finds significant terms to serve as an answer key.
4. Find out why `genes_in_list` in the `missMethyl` files can have fewer genes than `DE` (for example by comparing the symbols of the two annotation sources), and decide whether to map genes through Entrez identifiers instead of symbols.
5. After the first full run, write the numbers of Step 11 into the conclusions (Step 12) and the README.